# Talos - Dataset Preprocessing

Before training anything we clean the data from the logger and add the columns the models need.

This notebook makes two files:
- `data_clean.csv`: every 2-second row, cleaned, with extra columns
- `data_1min.csv`: the same data grouped per minute (used by the forecast and the AC recommender)

In [1]:
import pandas as pd
import numpy as np

## 1. Load the raw data

In [2]:
raw = pd.read_csv('data_final.csv')
raw['pc_time'] = pd.to_datetime(raw['pc_time'])
print(raw.shape)
raw.head()

(222452, 6)


,pc_time,uptime_ms,temp_c,humidity_pct,mic_peak,ir_cmd
0,2026-11-05 20:24:54,46255,27.3,77.6,322712,none
1,2026-11-05 20:24:56,48255,27.3,77.1,295990,none
2,2026-11-05 20:24:58,50255,27.3,76.7,652836,none
3,2026-11-05 20:25:00,52255,27.4,76.2,457246,none
4,2026-11-05 20:25:02,54255,27.4,75.9,374832,none


## 2. How much did the dataset grow from Assignment 2?

In Assignment 2 we only had one evening (about 4 hours, 6,508 rows) and the AC was on the whole time. For Assignment 3 we kept the logger running for more days, so now we also have daytime and AC-off periods.

In [3]:
days = raw['pc_time'].dt.date
summary = pd.DataFrame({
    'Assignment 2': ['6,508', 'about 4 hours (1 evening)', 'AC always on', '6'],
    'Assignment 3': [f'{len(raw):,}',
                     f"{(raw['pc_time'].max() - raw['pc_time'].min()).total_seconds() / 3600:.0f} hours ({days.nunique()} days)",
                     'AC on and off', 'see below'],
}, index=['Rows', 'Time covered', 'AC state', 'Columns'])
summary

,Assignment 2,Assignment 3
Rows,"6,508","222,452"
Time covered,about 4 hours (1 evening),124 hours (6 days)
AC state,AC always on,AC on and off
Columns,6,see below


In [4]:
raw.groupby(days).size().rename('rows per day')

pc_time
2026-11-05     6452
2026-11-06    43201
2026-11-07    43200
2026-11-08    43200
2026-11-09    43200
2026-11-10    43199
Name: rows per day, dtype: int64

## 3. Basic cleaning

- sort by time and drop exact duplicate rows
- remove values that are physically impossible (DHT22 errors)
- the mic maxes out at 5,000,000, we keep it but cap it there

In [5]:
df = raw.sort_values('pc_time').drop_duplicates().reset_index(drop=True)

ok = df['temp_c'].between(10, 45) & df['humidity_pct'].between(0, 100)
print('impossible rows removed:', (~ok).sum())
df = df[ok].reset_index(drop=True)

df['mic_peak'] = df['mic_peak'].clip(upper=5_000_000)
print('duplicates removed:', len(raw) - len(df) - (~ok).sum())

impossible rows removed: 0
duplicates removed: 0


## 4. Fix short spikes

Sometimes the DHT22 gives one weird value and then goes back to normal (like humidity 64 -> 82 -> 64). We use a rolling median over 5 rows (10 seconds) so these single spikes disappear, but real changes stay because they last longer than 10 seconds.

In [6]:
before = df[['temp_c', 'humidity_pct']].copy()
df['temp_c'] = df['temp_c'].rolling(5, center=True, min_periods=1).median()
df['humidity_pct'] = df['humidity_pct'].rolling(5, center=True, min_periods=1).median()
print('temp values changed:', (before['temp_c'] != df['temp_c']).sum())
print('humidity values changed:', (before['humidity_pct'] != df['humidity_pct']).sum())

temp values changed: 70200
humidity values changed: 129535


## 5. Find logging gaps

Normally there is one row every 2 seconds. If the next row comes more than 10 seconds later, something stopped (laptop off, board restart), so we mark it.

In [7]:
step = df['pc_time'].diff().dt.total_seconds()
df['gap_before'] = (step > 10).astype(int)
print('gaps found:', df['gap_before'].sum())
df.loc[df['gap_before'] == 1, ['pc_time']].assign(gap_seconds=step[df['gap_before'] == 1])

gaps found: 2


,pc_time,gap_seconds
1833,2026-11-05 21:26:35,36.0
4794,2026-11-05 23:05:33,53.0


## 6. Add new columns

**AC state.** `ir_cmd` is only filled when a command is sent, so we forward fill it to know the AC state on every row.

**Occupancy.** Same idea as the firmware: median of the mic over the last 5 minutes (150 rows), occupied if >= 30,000.

**Time.** Hour and day of week, because the AC routine depends a lot on the time.

In [8]:
cmd = df['ir_cmd'].where(df['ir_cmd'] != 'none').ffill()
df['ac_on'] = cmd.str.startswith('on').fillna(False).astype(int)
df['setpoint'] = cmd.str.extract(r'/(\d+)C/')[0].astype(float).fillna(21)
df['fan'] = cmd.str.split('/').str[3].fillna('max')

df['mic_median_5m'] = df['mic_peak'].rolling(150, min_periods=150).median()
df['activity'] = (df['mic_peak'] >= 2.0 * df['mic_median_5m']).astype(int)
df['occupied'] = df['activity'].rolling(450, min_periods=1).max().astype(int)

df['hour'] = df['pc_time'].dt.hour
df['weekday'] = df['pc_time'].dt.weekday   # 0 = Monday

df = df.dropna(subset=['mic_median_5m']).reset_index(drop=True)  # first 5 minutes have no median yet
df.head()

C:\Users\ryuu\AppData\Local\Temp\ipykernel_28984\1968103258.py:11: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['weekday'] = df['pc_time'].dt.weekday   # 0 = Monday


,pc_time,uptime_ms,temp_c,humidity_pct,mic_peak,ir_cmd,gap_before,ac_on,setpoint,fan,mic_median_5m,activity,occupied,hour,weekday
0,2026-11-05 20:29:52,344255,26.3,76.7,322780,none,0,1,16.0,fan_max,392715.5,0,0,20,3
1,2026-11-05 20:29:54,346255,26.3,76.4,262070,none,0,1,16.0,fan_max,392715.5,0,0,20,3
2,2026-11-05 20:29:56,348255,26.3,76.2,265683,none,0,1,16.0,fan_max,392715.5,0,0,20,3
3,2026-11-05 20:29:58,350255,26.3,76.1,361140,none,0,1,16.0,fan_max,386574.5,0,0,20,3
4,2026-11-05 20:30:00,352255,26.2,76.1,379653,none,0,1,16.0,fan_max,380395.0,0,0,20,3


## 7. Per-minute version

For the forecast and the AC recommender, every 2 seconds is too much. We group it per minute.

In [9]:
m = df.set_index('pc_time').resample('1min').agg({
    'temp_c': 'mean',
    'humidity_pct': 'mean',
    'mic_peak': 'median',
    'mic_median_5m': 'last',
    'occupied': 'last',
    'ac_on': 'last',
    'setpoint': 'last',
    'hour': 'last',
    'weekday': 'last',
}).dropna()

m['temp_change_5m'] = m['temp_c'] - m['temp_c'].shift(5)       # getting hotter or colder?
m['temp_30m_ago'] = m['temp_c'].shift(30)
# AC setting as one label: off, 16C, 17C ... 21C
m['ac_setting'] = np.where(m['ac_on'] == 1, m['setpoint'].astype(int).astype(str) + 'C', 'off')
m = m.round(3)
print(m.shape)
m.head()

(7411, 12)


,temp_c,humidity_pct,mic_peak,mic_median_5m,occupied,ac_on,setpoint,hour,weekday,temp_change_5m,temp_30m_ago,ac_setting
pc_time,,,,,,,,,,,,
2026-11-05 20:29:00,26.300,76.350,294231.5,386574.5,0,1,16.0,20,3,NaN,NaN,16C
2026-11-05 20:30:00,26.150,75.027,355310.5,394682.0,0,1,16.0,20,3,NaN,NaN,16C
2026-11-05 20:31:00,25.977,74.177,260611.5,358381.5,1,1,16.0,20,3,NaN,NaN,16C
2026-11-05 20:32:00,25.850,73.890,128911.0,303576.5,1,1,16.0,20,3,NaN,NaN,16C
2026-11-05 20:33:00,25.753,73.097,121071.0,237622.5,1,1,16.0,20,3,NaN,NaN,16C


## 8. Save

In [10]:
df.to_csv('data_clean.csv', index=False)
m.to_csv('data_1min.csv')
print('data_clean.csv:', df.shape)
print('data_1min.csv :', m.shape)
print(list(df.columns))

data_clean.csv: (222303, 15)
data_1min.csv : (7411, 12)
['pc_time', 'uptime_ms', 'temp_c', 'humidity_pct', 'mic_peak', 'ir_cmd', 'gap_before', 'ac_on', 'setpoint', 'fan', 'mic_median_5m', 'activity', 'occupied', 'hour', 'weekday']
